In [1]:
%%bash

pip install haystack-ai
pip install "datasets>=2.6.1"
pip install "sentence-transformers>=4.1.0"


Defaulting to user installation because normal site-packages is not writeable


Defaulting to user installation because normal site-packages is not writeable


Defaulting to user installation because normal site-packages is not writeable


In [2]:
from haystack.document_stores.in_memory import InMemoryDocumentStore

document_store = InMemoryDocumentStore()


In [3]:
from datasets import load_dataset
from haystack import Document

dataset = load_dataset("bilgeyucel/seven-wonders", split="train")
docs = [Document(content=doc["content"], meta=doc["meta"]) for doc in dataset]


/home/chengxi/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
from haystack.components.embedders import SentenceTransformersDocumentEmbedder

doc_embedder = SentenceTransformersDocumentEmbedder(model="sentence-transformers/all-MiniLM-L6-v2")
doc_embedder.warm_up()


/home/chengxi/.local/lib/python3.10/site-packages/huggingface_hub/file_download.py:942: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [5]:
docs_with_embeddings = doc_embedder.run(docs)
document_store.write_documents(docs_with_embeddings["documents"])


Batches: 100%|██████████| 5/5 [00:01<00:00,  4.38it/s]


151

In [6]:
from haystack.components.embedders import SentenceTransformersTextEmbedder

text_embedder = SentenceTransformersTextEmbedder(model="sentence-transformers/all-MiniLM-L6-v2")


In [7]:
from haystack.components.retrievers.in_memory import InMemoryEmbeddingRetriever

retriever = InMemoryEmbeddingRetriever(document_store)


In [8]:
from haystack.components.builders import ChatPromptBuilder
from haystack.dataclasses import ChatMessage

template = [
    ChatMessage.from_user(
        """
Given the following information, answer the question.

Context:
{% for document in documents %}
    {{ document.content }}
{% endfor %}

Question: {{question}}
Answer:
"""
    )
]

prompt_builder = ChatPromptBuilder(template=template)


ChatPromptBuilder has 2 prompt variables, but `required_variables` is not set. By default, all prompt variables are treated as optional, which may lead to unintended behavior in multi-branch pipelines. To avoid unexpected execution, ensure that variables intended to be required are explicitly set in `required_variables`.


In [9]:
import os
from haystack.components.generators.chat import OpenAIChatGenerator


os.environ["OPENAI_API_KEY"] = "sk-IWcHVBG9f1UFOHYM3vyXBUJWgQfcbwH7KahozVAYFlLfXkO4"
os.environ["OPENAI_BASE_URL"] = "http://43.159.131.233:3001/v1"
# 2️⃣ 创建 Chat Generator
chat_generator = OpenAIChatGenerator(
    model="gpt-5",   # ⚠️ 确保你的后端支持该 model id
    timeout=60,
)



In [10]:
from haystack import Pipeline

basic_rag_pipeline = Pipeline()
# Add components to your pipeline
basic_rag_pipeline.add_component("text_embedder", text_embedder)
basic_rag_pipeline.add_component("retriever", retriever)
basic_rag_pipeline.add_component("prompt_builder", prompt_builder)
basic_rag_pipeline.add_component("llm", chat_generator)



In [11]:
# Now, connect the components to each other
basic_rag_pipeline.connect("text_embedder.embedding", "retriever.query_embedding")
basic_rag_pipeline.connect("retriever", "prompt_builder")
basic_rag_pipeline.connect("prompt_builder.prompt", "llm.messages")


🚅 Components
  - text_embedder: SentenceTransformersTextEmbedder
  - retriever: InMemoryEmbeddingRetriever
  - prompt_builder: ChatPromptBuilder
  - llm: OpenAIChatGenerator
🛤️ Connections
  - text_embedder.embedding -> retriever.query_embedding (list[float])
  - retriever.documents -> prompt_builder.documents (list[Document])
  - prompt_builder.prompt -> llm.messages (list[ChatMessage])

In [12]:
question = "display BGP device global configuration"

response = basic_rag_pipeline.run({"text_embedder": {"text": question}, "prompt_builder": {"question": question}})

print(response["llm"]["replies"][0].text)


Batches: 100%|██████████| 1/1 [00:00<00:00, 76.03it/s]


display current-configuration | begin bgp


## RAG 180 Q&A

In [ ]:
import json
import hashlib
from pathlib import Path

from haystack import Document
from haystack.document_stores.in_memory import InMemoryDocumentStore
from haystack.components.embedders import SentenceTransformersDocumentEmbedder
from haystack.document_stores.types import DuplicatePolicy

# 1) Load JSONL
path = Path("/data/chengxi/Haystack/rag-router.txt")
docs = []

for i, line in enumerate(path.read_text(encoding="utf-8").splitlines(), start=1):
    line = line.strip()
    if not line:
        continue

    obj = json.loads(line)  # each line is a JSON object: {"Question": "...", "Answer": "..."}
    q = (obj.get("Question") or "").strip()
    a = (obj.get("Answer") or "").strip()
    if not q or not a:
        continue

    # 2) Build Document content
    content = f"User question: {q}\nCLI command: {a}"

    # 3) Generate a stable unique ID (content hash)
    # - same QA pair => same id => will NOT be inserted twice
    # - new QA pair  => new id => inserted as new data
    doc_id = hashlib.sha1(content.encode("utf-8")).hexdigest()

    docs.append(
        Document(
            id=doc_id,
            content=content,
            meta={
                "source": "rag-router.txt",
                "row": i,
                "type": "qa_cli_mapping",
                "verb": a.split(maxsplit=1)[0],
                "question": q,
                "answer": a,
            },
        )
    )

# 4) Write to DocumentStore (insert-only semantics)
store = InMemoryDocumentStore()

# 如果你希望“重复就报错”，用 FAIL（默认也是 FAIL）
# store.write_documents(docs, policy=DuplicatePolicy.FAIL)

# 更常见：重复就跳过（实现“只新增”且可重复运行导入脚本）
store.write_documents(docs, policy=DuplicatePolicy.SKIP)

# 5) Embed and write back
embedder = SentenceTransformersDocumentEmbedder(model="sentence-transformers/all-MiniLM-L6-v2")
embedder.warm_up()

embedded_docs = embedder.run(documents=docs)["documents"]

# 这里用 OVERWRITE 的含义是：给同一条 Document(id 相同)补全 embedding 字段
# 不会把“不同 id 的历史数据”覆盖掉
store.write_documents(embedded_docs, policy=DuplicatePolicy.OVERWRITE)

print(f"Prepared {len(docs)} docs from file.")
print(f"DocumentStore now holds {store.count_documents()} documents.")
